---
title: Nonlinear Modelling
---

## Nonlinear Programs
While we have already seen examples of linear, quadratic and conic programs,
JuMP also supports other general smooth nonlinear (convex and nonconvex) optimization problems.

In [ ]:
using JuMP, Ipopt
model = Model(optimizer_with_attributes(Ipopt.Optimizer, "print_level" => 0));

## MLE using JuMP

Since we already have a bit of JuMP experience at this point,
let's try a modelling example and apply what we have learnt.
In this example, we compute the maximum likelihood estimate (MLE) of
the parameters of a Gaussian distribution i.e. the sample mean and variance.

If $X_{1}, \ldots, X_{n}$ are an id sample from a population with pdf or pmf
$f\left(x | \theta_{1}, \ldots, \theta_{k}\right),$ the likelihood function is defined by

$$
L(\theta | \mathbf{x})=L\left(\theta_{1}, \ldots, \theta_{k} | x_{1}, \ldots, x_{n}\right)=\prod_{i=1}^{n} f\left(x_{i} | \theta_{1}, \ldots, \theta_{k}\right)
$$

For each sample point $\mathbf{x}$, let $\hat{\theta}(\mathbf{x})$ be a parameter value
at which $L(\theta | \mathbf{x})$ attains its maximum as a function of $\theta,$ with $\mathbf{x}$ held fixed.
A maximum likelihood estimator (MLE) of the parameter $\theta$ based on a sample $\mathbf{X}$ is
$\hat{\theta}(\mathbf{X})$.

The Gaussian likelihood is -

$$
L(\theta | \mathbf{x})=\prod_{i=1}^{n} \frac{1}{(2 \pi)^{1 / 2}} e^{-(1 / 2)\left(x_{i}-\theta\right)^{2}}=\frac{1}{(2 \pi)^{n / 2}} e^{(-1 / 2) \Sigma_{i=1}^{n}\left(x_{i}-\theta\right)^{2}}
$$

In most cases, the natural logarithm of
$L(\theta | \mathbf{x}), \log L(\theta | \mathbf{x})$ (known as the log likelihood),
is used rather than $L(\theta | \mathbf{x})$ directly.
The reason is that the log likelihood is easier to differentiate.
This substituion is possible because the log function is strictly increasing on $(0, \infty)$,
which implies that the extrema of $L(\theta | \mathbf{x})$ and $\log L(\theta | \mathbf{x})$ coincide.

In [ ]:
using Random, Statistics

Random.seed!(1234)

n = 1_000
data = randn(n)

mle = Model(optimizer_with_attributes(Ipopt.Optimizer, "print_level" => 0))
@NLparameter(mle, problem_data[i = 1:n] == data[i])
μ0 = randn()
σ0 = rand() + 1
@info "Starting guess, mean: $μ0, std: $σ0"
@variable(mle, μ, start = μ0)
@variable(mle, σ >= 0.0, start = σ0)

In [ ]:
@NLexpression(mle, loglikelihood,
    -(n / 2) * (log(2π) + 2 * log(σ)) - inv(2 * σ^2) * sum((xi - μ)^2 for xi in problem_data)
)
@NLobjective(mle, Max, loglikelihood)

optimize!(mle)

println("μ = ", value(μ))
println("mean(data) = ", mean(data))
println("σ^2 = ", value(σ)^2)
println("var(data) = ", var(data))
println("MLE value: ", exp(objective_value(mle)))

In [ ]:
# Changing the data

data = randn(n)
optimize!(mle)

println("μ = ", value(μ))
println("mean(data) = ", mean(data))
println("σ^2 = ", value(σ)^2)
println("var(data) = ", var(data))
println("MLE objective: ", objective_value(mle))